# 01 — FP vs ArGEnT vs PointNetMLPJoint

This notebook performs the paper-facing **validation-split evaluation** for the main edge-representation comparison. It reconstructs checkpoints dynamically from the repository, validates local HDF5 assets, runs inference on a deterministic geometry-level holdout, and saves comparison artifacts under `Comparison/results/01_fp_vs_argent`.

**Scope**
- Regimes: `Uniform + Edge`, `Zonal + Edge`
- Families: `ArGEnT_self_att_noSDF`, `PointNetMLPJoint`, `PointNetMLPJoint_FP`
- Shared evaluation split: seed `42`, fraction `0.20`

In [ ]:
from __future__ import annotations
import ast, hashlib, importlib.util, json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
try:
    import torch
    import h5py
except ImportError as exc:
    raise RuntimeError('Install torch and h5py in the selected notebook kernel before executing this comparison.') from exc

_clean_kernel_guard = {'checkpoint_report', 'selected_checkpoints', 'all_samples', 'split_records', 'node_results', 'pooled_metrics', 'geometry_metrics'}
_preexisting = sorted(name for name in _clean_kernel_guard if name in globals())
if _preexisting:
    raise RuntimeError(f'Run this notebook from a clean kernel; found pre-existing globals: {_preexisting}')

CURRENT_DIR = Path.cwd()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'Uniform').exists() else CURRENT_DIR.parent
if not (REPO_ROOT / 'Uniform').exists():
    raise RuntimeError(f'Repository root not found from {CURRENT_DIR}')
COMPARISON_DIR = REPO_ROOT / 'Comparison'
RESULTS_DIR = COMPARISON_DIR / 'results' / '01_fp_vs_argent'
FIGURES_DIR = RESULTS_DIR / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(COMPARISON_DIR))
import eval_helpers as eh

SPLIT_SEED, EVAL_FRACTION = 42, 0.20
QUALITATIVE_EXAMPLE_PATH = COMPARISON_DIR / 'Examples' / 'disc_example_edge_deriv_zonal.h5'
EVALUATION_LABEL = 'validation-split evaluation'
FAMILIES = ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint', 'PointNetMLPJoint_FP']
DATASETS = {
    'Uniform': REPO_ROOT / 'Data_gen' / 'output' / 'disc_dataset_edge_deriv_uniform.h5',
    'Zonal': REPO_ROOT / 'Data_gen' / 'output' / 'disc_dataset_edge_deriv_zonal.h5',
}
COMMIT = __import__('subprocess').check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
VERSIONS = {
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'matplotlib': plt.matplotlib.__version__,
    'torch': getattr(torch, '__version__', 'unknown'),
    'h5py': getattr(h5py, '__version__', 'unknown'),
}

display(Markdown(
    f'**Commit:** `{COMMIT}`\n\n'
    f'**Results directory:** `{RESULTS_DIR}`\n\n'
    f'**Evaluation label:** **{EVALUATION_LABEL}**'
))

## Checkpoint discovery and compatibility audit

This cell discovers checkpoints dynamically under `Uniform/Edge/*/Trained_models` and `Zonal/Edge/*/Trained_models`, validates required architecture/normalization fields, and selects at most one checkpoint per `(regime, family)` without silently substituting incompatible candidates.

In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def decode(value):
    return value.decode() if isinstance(value, bytes) else value


def _required_checkpoint_status(payload):
    required = ['arch', 'model_state', 'target_mean', 'target_std', 'coord_center', 'coord_half_range']
    missing = [key for key in required if key not in payload]
    if missing:
        return f'skipped: missing keys: {missing}'

    arch = payload['arch']
    if not isinstance(arch, dict):
        return f'skipped: arch must be dict-like, got {type(arch).__name__}'

    try:
        target_mean = np.asarray(payload['target_mean'], dtype='float32').reshape(-1)
        target_std = np.asarray(payload['target_std'], dtype='float32').reshape(-1)
        coord_center = np.asarray(payload['coord_center'], dtype='float32').reshape(-1)
        coord_half = np.asarray(payload['coord_half_range'], dtype='float32').reshape(-1)
    except Exception as exc:
        return f'skipped: normalization parsing failed: {type(exc).__name__}: {exc}'

    if target_mean.size not in (1, 2) or target_std.size != target_mean.size:
        return f'skipped: unexpected target normalization shapes mean={target_mean.shape} std={target_std.shape}'
    if coord_center.size != 2 or coord_half.size != 2:
        return f'skipped: expected 2D coordinate normalization, got center={coord_center.shape} half={coord_half.shape}'
    if np.any(~np.isfinite(target_mean)) or np.any(~np.isfinite(target_std)):
        return 'skipped: target normalization contains non-finite values'
    if np.any(~np.isfinite(coord_center)) or np.any(~np.isfinite(coord_half)):
        return 'skipped: coordinate normalization contains non-finite values'
    return 'ready'


def discover_checkpoints():
    rows = []
    for regime in DATASETS:
        for family in FAMILIES:
            folder = REPO_ROOT / regime / 'Edge' / family
            scripts = sorted(folder.glob('Training_script*.py')) if folder.exists() else []
            checkpoints = sorted((folder / 'Trained_models').glob('*.pt')) if folder.exists() else []
            if not folder.exists():
                rows.append({
                    'regime': regime,
                    'ablation': 'Edge',
                    'model_family': family,
                    'status': 'missing family directory',
                    'selected': False,
                    'checkpoint_path': None,
                    'training_scripts': [str(x) for x in scripts],
                })
                continue
            if not checkpoints:
                rows.append({
                    'regime': regime,
                    'ablation': 'Edge',
                    'model_family': family,
                    'status': 'missing checkpoint',
                    'selected': False,
                    'checkpoint_path': None,
                    'training_scripts': [str(x) for x in scripts],
                })
                continue
            for path in checkpoints:
                payload = None
                status = 'ready'
                metadata = {}
                try:
                    payload = torch.load(path, map_location='cpu', weights_only=False)
                    status = _required_checkpoint_status(payload)
                    metadata = {
                        'arch': payload.get('arch'),
                        'model_name': payload.get('model_name'),
                        'best_val_loss': payload.get('best_val_loss'),
                        'extra_feat_cols': payload.get('extra_feat_cols', []),
                        'target_names': payload.get('target_names'),
                        'h5_filename': payload.get('h5_filename'),
                        'representation': payload.get('representation'),
                    }
                except Exception as exc:
                    status = f'skipped: {type(exc).__name__}: {exc}'
                rows.append({
                    'regime': regime,
                    'ablation': 'Edge',
                    'model_family': family,
                    'status': status,
                    'selected': False,
                    'checkpoint_path': str(path),
                    'file_size_bytes': path.stat().st_size,
                    'sha256': sha256(path),
                    'training_scripts': [str(x) for x in scripts],
                    **metadata,
                })
    report = pd.DataFrame(rows)
    if report.empty:
        raise RuntimeError('No checkpoint candidates were discovered.')

    for (regime, family), group in report.groupby(['regime', 'model_family'], dropna=False):
        ready = group[group['status'].eq('ready')].copy()
        if ready.empty:
            continue
        if ready['best_val_loss'].notna().any():
            best_idx = ready['best_val_loss'].astype(float).idxmin()
        else:
            best_idx = ready.sort_values('checkpoint_path').index[0]
        report.loc[best_idx, 'selected'] = True
        extra_idx = [idx for idx in ready.index if idx != best_idx]
        if extra_idx:
            report.loc[extra_idx, 'status'] = 'skipped: additional ready checkpoint candidate'
            report.loc[extra_idx, 'selected'] = False

    report = report.sort_values(['regime', 'model_family', 'checkpoint_path'], na_position='last').reset_index(drop=True)
    (RESULTS_DIR / 'checkpoint_integrity.json').write_text(report.to_json(orient='records', indent=2), encoding='utf-8')
    selected = report[report['selected']].copy().reset_index(drop=True)
    eh.save_table(selected[['regime', 'ablation', 'model_family', 'checkpoint_path', 'sha256', 'best_val_loss']], RESULTS_DIR, 'selected_checkpoints')
    return report, selected


checkpoint_report, selected_checkpoints = discover_checkpoints()
display(checkpoint_report[['regime', 'model_family', 'status', 'selected', 'checkpoint_path', 'sha256']])

## HDF5 loading, schema checks, and geometry-level split

The notebook requires local HDF5 assets in `Data_gen/output/`. It verifies the `edge` representation and constructs a deterministic geometry-level holdout with `seed=42` and `fraction=0.20`.

In [ ]:
def load_samples(path):
    samples = []
    with h5py.File(path, 'r') as h5:
        representation = decode(h5.attrs.get('representation', ''))
        if representation != 'edge':
            raise ValueError(f'{path.name}: expected representation edge, got {representation!r}')
        if 'samples' not in h5:
            raise ValueError(f'{path.name}: missing top-level group "samples"')
        for key in sorted(h5['samples'].keys()):
            g = h5['samples'][key]
            def arr(name, default=None):
                return np.asarray(g[name]) if name in g else default
            coords = arr('node_coords_mm')
            stress = arr('stress_max_vm')
            life = arr('life_raw')
            if coords is None or stress is None or life is None:
                raise ValueError(f'{path.name}/{key}: missing required target fields')
            if coords.ndim != 2 or coords.shape[1] != 2:
                raise ValueError(f'{path.name}/{key}: expected node_coords_mm shape [N,2], got {coords.shape}')
            sample_id = decode(g.attrs.get('sample_id', key))
            attrs = {str(k): decode(v) for k, v in g.attrs.items()}
            samples.append({
                'sample_key': key,
                'sample_id': str(sample_id),
                'attrs': attrs,
                'coords': coords.astype('float32'),
                'stress': stress.astype('float32').reshape(-1),
                'loglife': np.log10(np.clip(life.astype('float64').reshape(-1), 1e-30, None)).astype('float32'),
                'zone_id': arr('zone_id', np.full(len(coords), -1)).reshape(-1),
                'subzone_id': arr('subzone_id', np.full(len(coords), np.nan)).reshape(-1),
                'arc_length_mm': arr('arc_length_mm', np.arange(len(coords), dtype='float32')).reshape(-1),
                'node_features': arr('node_features', np.empty((len(coords), 0), dtype='float32')),
            })
    return samples


def split_samples(samples):
    rng = np.random.default_rng(SPLIT_SEED)
    order = rng.permutation(len(samples))
    n_eval = max(1, int(round(len(samples) * EVAL_FRACTION)))
    eval_pos = np.sort(order[:n_eval]).tolist()
    train_pos = np.sort(order[n_eval:]).tolist()
    return train_pos, eval_pos


all_samples, split_records = {}, {}
for regime, path in DATASETS.items():
    if not path.exists():
        raise FileNotFoundError(
            f'Missing required local asset for {regime}: {path}. '
            'Generate or copy the HDF5 before running this notebook.'
        )
    samples = load_samples(path)
    train_pos, eval_pos = split_samples(samples)
    eval_ids = [samples[i]['sample_id'] for i in eval_pos]
    if len(set(eval_ids)) != len(eval_ids):
        raise ValueError(f'{regime}: duplicate sample_id values detected in evaluation split')
    all_samples[regime] = samples
    split_records[regime] = {
        'hdf5_filename': path.name,
        'total_geometry_count': len(samples),
        'evaluation_geometry_count': len(eval_pos),
        'split_seed': SPLIT_SEED,
        'split_fraction': EVAL_FRACTION,
        'training_sample_ids': [samples[i]['sample_id'] for i in train_pos],
        'evaluation_sample_ids': eval_ids,
        'training_positional_indices': train_pos,
        'evaluation_positional_indices': eval_pos,
        'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'notebook_commit_sha': COMMIT,
        'evaluation_label': EVALUATION_LABEL,
        'independence_basis': 'A new geometry holdout is not proved independent of model validation/checkpoint selection.',
    }

with open(RESULTS_DIR / 'evaluation_split_provenance.json', 'w', encoding='utf-8') as stream:
    json.dump(split_records, stream, indent=2)

display(pd.DataFrame([
    {
        'regime': regime,
        'hdf5_filename': data['hdf5_filename'],
        'geometries': data['total_geometry_count'],
        'evaluation_geometries': data['evaluation_geometry_count'],
        'label': data['evaluation_label'],
    }
    for regime, data in split_records.items()
]))

## Model reconstruction, checkpoint validation, and shared-geometry inference

This section reconstructs each selected model using the checkpoint's own architecture, refuses incompatible FP fallbacks, runs inference on the holdout split, and then enforces a **shared geometry ID set across all three families** inside each regime.

In [ ]:
def import_local(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def reconstruct(row):
    folder = Path(row['checkpoint_path']).parent.parent
    family = row['model_family']
    ckpt = torch.load(row['checkpoint_path'], map_location='cpu', weights_only=False)
    arch = dict(ckpt['arch'])
    pn = import_local(folder / 'pn_models.py', f'pn_{row["regime"]}_{family}')
    sys.modules['pn_models'] = pn
    if family == 'PointNetMLPJoint_FP':
        if not hasattr(pn, 'build_fp_model_from_arch'):
            raise RuntimeError('FP checkpoint requires build_fp_model_from_arch; refusing regular PointNet fallback')
        model = pn.build_fp_model_from_arch(arch)
    elif family in ('PointNetMLPJoint', 'PointNetMLPJoint_weighted'):
        model = pn.build_model_from_arch(arch)
    else:
        bench = import_local(folder / 'benchmarks.py', f'bench_{row["regime"]}_{family}')
        if not hasattr(bench, 'ArGEnTDeepONet'):
            raise RuntimeError('No ArGEnTDeepONet found in benchmarks.py')
        defaults = {
            'hidden_dim': 128,
            'num_heads': 4,
            'num_layers': 2,
            'output_dim': 128,
            'out_channels': 1,
            'attention_type': 'self',
            'use_sdf': False,
            'in_ch_geom': 2,
        }
        defaults.update({k: v for k, v in arch.items() if k in defaults})
        if 'out_channels' not in arch and 'bias' in ckpt['model_state']:
            defaults['out_channels'] = int(ckpt['model_state']['bias'].shape[0])
        model = bench.ArGEnTDeepONet(**defaults)
    model.load_state_dict(ckpt['model_state'], strict=True)
    model.eval()
    return model, ckpt


def feature_matrix(sample, ckpt):
    cols = ckpt.get('extra_feat_cols', []) or []
    available = sample['node_features']
    if cols and available.shape[1] < len(cols):
        raise ValueError(f'missing required extra-feature columns: {cols}')
    extra = available[:, :len(cols)] if cols else np.empty((len(sample['coords']), 0), dtype='float32')
    center = np.asarray(ckpt['coord_center'], dtype='float32')
    half = np.asarray(ckpt['coord_half_range'], dtype='float32')
    if center.shape != (2,) or half.shape != (2,):
        raise ValueError(f'expected 2D normalization, got center={center.shape}, half={half.shape}')
    coords = (sample['coords'] - center) / np.maximum(half, 1e-8)
    if extra.shape[1]:
        stats = ckpt.get('extra_feat_stats')
        if stats is None:
            raise ValueError('missing extra-feature normalization statistics')
        if isinstance(stats, dict):
            mean = np.asarray(stats['mean'], dtype='float32')
            std = np.asarray(stats['std'], dtype='float32')
        else:
            mean = np.asarray(stats[0], dtype='float32')
            std = np.asarray(stats[1], dtype='float32')
        extra = (extra - mean) / np.maximum(std, 1e-8)
    return coords, extra


def predict(model, sample, ckpt):
    coords, extra = feature_matrix(sample, ckpt)
    x = torch.from_numpy(coords[None])
    q = x.clone()
    with torch.no_grad():
        try:
            out = model(x, q)
        except TypeError:
            out = model(torch.cat([x, torch.from_numpy(extra[None])], dim=-1), q)
    out = out.detach().cpu().numpy()
    if out.ndim != 3 or out.shape[0] != 1:
        raise ValueError(f'prediction shape unexpected: {out.shape}')
    mean = np.asarray(ckpt['target_mean'])
    std = np.asarray(ckpt['target_std'])
    out = out * std + mean
    if out.shape[2] == 2:
        return out[0, :, 0], out[0, :, 1]
    if out.shape[2] == 1:
        return np.zeros(out.shape[1], dtype='float32'), out[0, :, 0]
    raise ValueError(f'unexpected output channels: {out.shape[2]}')


required_per_regime = pd.MultiIndex.from_product([list(DATASETS.keys()), FAMILIES], names=['regime', 'model_family'])
selected_pairs = pd.MultiIndex.from_frame(selected_checkpoints[['regime', 'model_family']])
missing_pairs = [tuple(x) for x in required_per_regime.difference(selected_pairs).tolist()]
if missing_pairs:
    raise RuntimeError(f'Missing required compatible checkpoints: {missing_pairs}')

node_frames, inference_errors = [], []
for _, row in selected_checkpoints.sort_values(['regime', 'model_family']).iterrows():
    try:
        model, ckpt = reconstruct(row)
        for i in split_records[row['regime']]['evaluation_positional_indices']:
            sample = all_samples[row['regime']][i]
            pred_stress, pred_loglife = predict(model, sample, ckpt)
            if len(pred_stress) != len(sample['coords']) or len(pred_loglife) != len(sample['coords']):
                raise ValueError('prediction length mismatch relative to sample coordinates')
            base = pd.DataFrame({
                'regime': row['regime'],
                'ablation': 'Edge',
                'model_family': row['model_family'],
                'sample_key': sample['sample_key'],
                'sample_id': sample['sample_id'],
                'node_idx': np.arange(len(sample['coords'])),
                'x_mm': sample['coords'][:, 0],
                'r_mm': sample['coords'][:, 1],
                'zone_id': sample['zone_id'],
                'subzone_id': sample['subzone_id'],
                'arc_length_mm': sample['arc_length_mm'],
                'true_stress': sample['stress'],
                'pred_stress': pred_stress,
                'true_loglife': sample['loglife'],
                'pred_loglife': pred_loglife,
                'evaluation_label': EVALUATION_LABEL,
            })
            base['zone_name'] = base['zone_id'].map(eh.ZONE_ID_TO_NAME)
            base['subzone_name'] = base['subzone_id'].map(eh.SUBZONE_ID_TO_NAME)
            node_frames.append(base)
    except Exception as exc:
        inference_errors.append({
            'regime': row['regime'],
            'model_family': row['model_family'],
            'checkpoint_path': row['checkpoint_path'],
            'status': f'skipped during inference: {type(exc).__name__}: {exc}',
        })

if inference_errors:
    with open(RESULTS_DIR / 'inference_errors.json', 'w', encoding='utf-8') as stream:
        json.dump(inference_errors, stream, indent=2)
    raise RuntimeError(f'Inference failed for one or more required checkpoints: {inference_errors}')

node_results = pd.concat(node_frames, ignore_index=True) if node_frames else pd.DataFrame()
if node_results.empty:
    raise RuntimeError('No node-level inference results were produced.')

shared_geometry = {}
for regime in DATASETS:
    fam_to_ids = {}
    for family in FAMILIES:
        ids = sorted(node_results.loc[(node_results['regime'] == regime) & (node_results['model_family'] == family), 'sample_id'].astype(str).unique().tolist())
        if not ids:
            raise RuntimeError(f'{regime}: no inference results for required family {family}')
        fam_to_ids[family] = ids
    shared_ids = set(fam_to_ids[FAMILIES[0]])
    for family in FAMILIES[1:]:
        shared_ids &= set(fam_to_ids[family])
    shared_ids = sorted(shared_ids)
    if not shared_ids:
        raise RuntimeError(f'{regime}: the three families share no common geometry IDs')
    if any(ids != shared_ids for ids in fam_to_ids.values()):
        warnings.warn(f'{regime}: pruning to {len(shared_ids)} shared geometry IDs across all three families')
        node_results = node_results[(node_results['regime'] != regime) | (node_results['sample_id'].isin(shared_ids))].copy()
    shared_geometry[regime] = {
        'families': fam_to_ids,
        'shared_geometry_ids': shared_ids,
        'shared_geometry_count': len(shared_ids),
        'requested_evaluation_geometry_count': len(split_records[regime]['evaluation_sample_ids']),
        'evaluation_label': EVALUATION_LABEL,
    }

node_results = node_results.sort_values(['regime', 'model_family', 'sample_id', 'node_idx']).reset_index(drop=True)
# node_results DataFrame is kept in memory only (too large to save as CSV).
with open(RESULTS_DIR / 'shared_geometry_ids.json', 'w', encoding='utf-8') as stream:
    json.dump(shared_geometry, stream, indent=2)

display(node_results.groupby(['regime', 'model_family']).sample_id.nunique().reset_index(name='shared_evaluation_geometries'))

## Metrics, paired comparisons, and compact saved tables

The next cell computes pooled metrics, full LogLife life-band metrics, grouped physical-region metrics, and geometry-level whole-field/minimum-life diagnostics.


In [ ]:
pooled_metrics = eh.pooled_metrics_from_nodes(node_results)
life_band_metrics = eh.loglife_bin_metrics(node_results)
grouped_region_metrics = eh.grouped_region_metrics_from_nodes(node_results)
geometry_metrics = eh.geometry_level_metrics(node_results)

for frame in [pooled_metrics, life_band_metrics, grouped_region_metrics, geometry_metrics]:
    if not frame.empty:
        frame.insert(0, 'evaluation_label', EVALUATION_LABEL)

eh.save_table(pooled_metrics, RESULTS_DIR, 'pooled_metrics')
eh.save_table(life_band_metrics, RESULTS_DIR, 'life_band_metrics')
eh.save_table(grouped_region_metrics, RESULTS_DIR, 'grouped_region_metrics')

summary_rows = []
for (regime, family), _ in pooled_metrics[pooled_metrics['target'] == 'LogLife'].groupby(['regime', 'model_family']):
    row = {'evaluation_label': EVALUATION_LABEL, 'regime': regime,
           'model_family': family, 'ablation': 'Edge'}
    for target in ('Stress', 'LogLife'):
        sub = pooled_metrics[(pooled_metrics['regime'] == regime) &
                             (pooled_metrics['model_family'] == family) &
                             (pooled_metrics['target'] == target)]
        if not sub.empty:
            row[f'{target}_MAE'] = float(sub.iloc[0]['MAE'])
            row[f'{target}_RMSE'] = float(sub.iloc[0]['RMSE'])
            row[f'{target}_R2'] = float(sub.iloc[0]['R2 (log)'])
    sub_g = geometry_metrics[(geometry_metrics['regime'] == regime) & (geometry_metrics['model_family'] == family)]
    if not sub_g.empty:
        row['whole_geometry_mean_LogLife_MAE'] = float(sub_g['whole_geometry_loglife_mae'].mean())
        row['whole_geometry_mean_Stress_MAE'] = float(sub_g['whole_geometry_stress_mae'].mean())
        row['absolute_min_loglife_error_mean'] = float(sub_g['abs_min_loglife_error_decades'].mean())
        row['absolute_max_stress_error_mean'] = float(sub_g['abs_max_stress_error'].mean())
        row['critical_zone_agreement'] = float(sub_g['same_zone_critical'].mean())
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows).sort_values(['regime', 'model_family']).reset_index(drop=True)
eh.save_table(summary_table, RESULTS_DIR, 'summary_table')

geometry_summary = geometry_metrics.groupby(['regime', 'ablation', 'model_family'], as_index=False).agg(
    n_geometries=('sample_id', 'nunique'),
    whole_geometry_mean_loglife_mae=('whole_geometry_loglife_mae', 'mean'),
    whole_geometry_mean_stress_mae=('whole_geometry_stress_mae', 'mean'),
    absolute_min_loglife_error_mean=('abs_min_loglife_error_decades', 'mean'),
    absolute_max_stress_error_mean=('abs_max_stress_error', 'mean'),
    critical_zone_agreement=('same_zone_critical', 'mean'),
)
eh.save_table(geometry_summary, RESULTS_DIR, 'geometry_summary')

def paired_diff(geom_df, left_family, right_family):
    rows = []
    for regime, sub in geom_df.groupby('regime'):
        left = sub[sub['model_family'] == left_family].set_index('sample_id')
        right = sub[sub['model_family'] == right_family].set_index('sample_id')
        common = sorted(set(left.index).intersection(right.index))
        if not common:
            continue
        diffs = [
            float(left.loc[sid]['abs_min_loglife_error_decades'] - right.loc[sid]['abs_min_loglife_error_decades'])
            for sid in common
        ]
        s = pd.Series(diffs)
        rows.append({
            'evaluation_label': EVALUATION_LABEL, 'regime': regime,
            'left_family': left_family, 'right_family': right_family,
            'n_geometries': len(common),
            'median_left_minus_right_abs_min_loglife_err': float(s.median()),
            'fraction_right_better': float((s > 0).mean()),
        })
    return pd.DataFrame(rows)

paired_summary = pd.concat([
    paired_diff(geometry_metrics, 'ArGEnT_self_att_noSDF', 'PointNetMLPJoint_FP'),
    paired_diff(geometry_metrics, 'PointNetMLPJoint', 'PointNetMLPJoint_FP'),
], ignore_index=True)
eh.save_table(paired_summary, RESULTS_DIR, 'paired_summary')

run_metadata = {
    'commit_sha': COMMIT, 'software_versions': VERSIONS,
    'evaluation_label': EVALUATION_LABEL,
    'results_dir': str(RESULTS_DIR),
    'families': FAMILIES,
    'datasets': {k: str(v) for k, v in DATASETS.items()},
    'quantitative_only_from_production_hdf5': True,
    'qualitative_examples_use_example_hdf5_only': True,
}
eh.save_json(run_metadata, RESULTS_DIR, 'run_metadata')

display(summary_table)
display(paired_summary)


## Representative geometries

Representative geometries are selected per regime from the shared evaluated IDs:
- **median** composite behaviour,
- **lowest true minimum life**,
- **greatest ArGEnT–FP disagreement** in absolute minimum-life error.

In [ ]:
representatives = {}
for regime in DATASETS:
    picks = eh.select_representative_geometries(
        geometry_metrics, regime, 'Edge',
        disagreement_family='PointNetMLPJoint_FP',
    )
    # Keep only most_critical and median for compact output
    representatives[regime] = {k: v for k, v in picks.items() if k in ('most_critical', 'median')}

eh.save_json({'evaluation_label': EVALUATION_LABEL, 'representatives': representatives},
             RESULTS_DIR, 'representative_geometry_ids')
display(representatives)


## Figures

This cell writes compact quantitative figures only (no production-HDF5 qualitative maps):
- life-band comparison,
- grouped-region comparison,
- per-geometry whole-field vs absolute-minimum-life distributions.


In [ ]:
for regime in DATASETS:
    regime_fig_dir = FIGURES_DIR / regime.lower()
    regime_fig_dir.mkdir(parents=True, exist_ok=True)
    sub_bins = life_band_metrics[life_band_metrics['regime'] == regime].copy()
    sub_groups = grouped_region_metrics[(grouped_region_metrics['regime'] == regime) & (grouped_region_metrics['status'] != 'missing')].copy()
    sub_geom = geometry_metrics[geometry_metrics['regime'] == regime].copy()

    eh.plot_bin_bar(
        sub_bins,
        f'{regime} life-band LogLife metrics ({EVALUATION_LABEL})',
        out_dir=regime_fig_dir,
        filename='figure_life_bands',
    )

    if not sub_groups.empty:
        regions = list(eh.GROUPED_REGION_DEFS.keys())
        fig, ax = plt.subplots(figsize=(10, 4.5))
        x = np.arange(len(regions), dtype=float)
        width = 0.8 / max(1, len(FAMILIES))
        for i, fam in enumerate(FAMILIES):
            d = sub_groups[sub_groups['model_family'] == fam].set_index('grouped_region')
            vals = [d.loc[r, 'LogLife_MAE'] if r in d.index else np.nan for r in regions]
            ax.bar(x + (i - (len(FAMILIES)-1)/2)*width, vals, width=width, label=fam)
        ax.set_xticks(x); ax.set_xticklabels(regions, rotation=20, ha='right')
        ax.set_ylabel('LogLife MAE (decades)')
        ax.set_title(f'{regime} grouped physical-region metrics')
        ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(regime_fig_dir / 'figure_grouped_regions.png', dpi=150, bbox_inches='tight')
        fig.savefig(regime_fig_dir / 'figure_grouped_regions.pdf', bbox_inches='tight')
        plt.close(fig)

    eh.plot_geometry_error_distributions(
        sub_geom,
        f'{regime} geometry-level error distributions ({EVALUATION_LABEL})',
        out_dir=regime_fig_dir,
        filename='figure_geometry_distributions',
    )
    plt.close('all')


*Artifacts saved under `Comparison/results/01_fp_vs_argent/`.*


In [ ]:
# ── Regular-model inference illustration ─────────────────────────────────────
# Show the qualitative inference field for all regular-model families using
# the Zonal checkpoints. The figure is saved with equal aspect ratio so
# the disc geometry reads correctly (not as an elongated I-beam shape).
#
# Scope: Zonal / Edge regime only (this is also the basis for the other notebooks).
# For each selected model family, the prediction is made on the first sample in
# the example HDF5 file.  This is purely qualitative; quantitative comparisons
# use the geometry-level validation split (see eval cells above).

if QUALITATIVE_EXAMPLE_PATH.exists():
    ex_samples = load_samples(QUALITATIVE_EXAMPLE_PATH)
    if ex_samples:
        ex = ex_samples[0]
        by_model = {}
        # Use Zonal regime selected checkpoints for the illustration
        zonal_selected = selected_checkpoints[selected_checkpoints['regime'] == 'Zonal']
        for _, row in zonal_selected.sort_values('model_family').iterrows():
            try:
                model, ckpt = reconstruct(row)
                pred_stress, pred_loglife = predict(model, ex, ckpt)
                f = pd.DataFrame({
                    'x_mm':         ex['coords'][:, 0],
                    'r_mm':         ex['coords'][:, 1],
                    'true_loglife': ex['loglife'],
                    'pred_loglife': pred_loglife,
                    'true_stress':  ex['stress'],
                    'pred_stress':  pred_stress,
                })
                by_model[row['model_family']] = f
            except Exception as exc:
                warnings.warn(f"Could not generate illustration for {row['model_family']}: {exc}")
        if by_model:
            illus_dir = FIGURES_DIR / 'zonal'
            illus_dir.mkdir(parents=True, exist_ok=True)
            fig = eh.plot_field_comparison(
                by_model, 'true_loglife', 'pred_loglife', 'decades',
                (
                    'Regular-model inference illustration — Zonal / Edge regimemodels '
                    '(ArGEnT, PointNetMLPJoint, PointNetMLPJoint_FP).\n'
                    'Qualitative example from the same FEM/data-generation pipeline; '
                    'quantitative comparisons use the geometry-level validation split.'
                ),
                out_dir=illus_dir,
                filename='figure_qualitative_example_regular',
            )
            plt.close('all')
            display(Markdown(
                f'**Regular-model illustration saved** — '
                f'`{illus_dir}/figure_qualitative_example_regular.png`'
            ))
        else:
            display(Markdown('No Zonal checkpoints were available for the illustration.'))
    else:
        display(Markdown(f'Example HDF5 at `{QUALITATIVE_EXAMPLE_PATH}` is empty.'))
else:
    display(Markdown(
        f'**Qualitative example file not found:** `{QUALITATIVE_EXAMPLE_PATH}`.\n\n'
        'To enable the illustration, generate or copy a single-sample HDF5 file '
        '(edge-deriv representation, Zonal geometry) to `Comparison/Examples/`.\n\n'
        f'Expected output: `{FIGURES_DIR}/zonal/figure_qualitative_example_regular.png`'
    ))


In [ ]:
# Artifacts saved to RESULTS_DIR; see run_metadata.json for provenance.
import os
artifacts = sorted(str(p.relative_to(RESULTS_DIR)) for p in RESULTS_DIR.rglob("*") if p.is_file())
display(pd.DataFrame({"artifact": artifacts}))


## Conclusion

This notebook provides a **validation-split evaluation** (not an independent test set) of
`PointNetMLPJoint_FP`, `PointNetMLPJoint`, and `ArGEnT_self_att_noSDF` on the
`Uniform/Edge` and `Zonal/Edge` regimes.

It reports:
- whole-field pooled metrics,
- full LogLife life-band metrics,
- grouped physical-region metrics,
- geometry-level whole-field MAE and absolute minimum-life error distributions.

Qualitative illustrative examples are reserved for local example HDF5 assets; quantitative rankings use only the deterministic geometry-level validation split from production HDF5.
